# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get all RecordSets by their @id
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = [rs['@id'] for rs in metadata.record_sets]
elif hasattr(metadata, 'recordSet'):
    # Some schemas use 'recordSet' as an attribute (singular/plural)
    rs_obj = getattr(metadata, 'recordSet')
    # May be list of dicts or list of objects with '@id'
    if isinstance(rs_obj, (list, tuple)):
        if len(rs_obj) > 0 and isinstance(rs_obj[0], dict) and '@id' in rs_obj[0]:
            record_sets = [x['@id'] for x in rs_obj]
        elif len(rs_obj) > 0 and hasattr(rs_obj[0], '@id'):
            record_sets = [getattr(x, '@id') for x in rs_obj]
    # If it's a dict with '@id'
    elif isinstance(rs_obj, dict) and '@id' in rs_obj:
        record_sets = [rs_obj['@id']]

if not record_sets:
    print("No record sets found in the metadata. Please inspect metadata to determine available data.")
else:
    print(f"Available record sets (@id): {record_sets}")

# For each record set, print overview of fields (by @id)
for record_set_id in record_sets:
    print(f"\nRecord Set: {record_set_id}")
    try:
        recset = dataset.schema.find_by_id(record_set_id)
        if recset is None:
            continue
        if hasattr(recset, 'fields'):
            field_ids = [field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', str(field)) for field in recset.fields]
            print(f"  Fields (@id): {field_ids}")
        elif hasattr(recset, 'field'):
            field = recset.field
            # field may be list of dicts or list of strings
            if isinstance(field, (list, tuple)):
                field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else getattr(f, '@id', str(f)) for f in field]
            else:
                field_ids = [field['@id']] if isinstance(field, dict) and '@id' in field else [field]
            print(f"  Fields (@id): {field_ids}")
        else:
            print("  No field information available.")
        # List columns if present
        if hasattr(recset, 'columns'):
            col_ids = [col['@id'] if isinstance(col, dict) and '@id' in col else getattr(col, '@id', str(col)) for col in recset.columns]
            print(f"  Columns (@id): {col_ids}")
        elif hasattr(recset, 'column'):
            col = recset.column
            if isinstance(col, (list, tuple)):
                col_ids = [c['@id'] if isinstance(c, dict) and '@id' in c else getattr(c, '@id', str(c)) for c in col]
            else:
                col_ids = [col['@id']] if isinstance(col, dict) and '@id' in col else [col]
            print(f"  Columns (@id): {col_ids}")
    except Exception as e:
        print(f"  Could not extract field info for {record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Demonstrate extraction process for all record sets
dataframes = {}
if not record_sets:
    print("No record sets to extract.")
else:
    for record_set_id in record_sets:
        try:
            records_iter = dataset.records(record_set=record_set_id)
            records = list(records_iter)
            if len(records) == 0:
                print(f"No records found for {record_set_id}")
                continue
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded {len(df)} records for {record_set_id}")
            print("Fields/columns available:", df.columns.tolist())
            print(df.head())
        except Exception as e:
            print(f"Error loading records for record set {record_set_id}: {e}")

# For demonstration, select the first available record set for further EDA
selected_record_set_id = record_sets[0] if record_sets else None
if selected_record_set_id and selected_record_set_id in dataframes:
    print(f"\nSelecting {selected_record_set_id} for EDA.")
    print(dataframes[selected_record_set_id].info())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Define field ids to use for EDA ---
# To ensure portability, list all column names for the selected record set

if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Attempt to select a likely numeric field (by column name)
    # You can change these based on the overview output!
    possible_numeric_fields = ['cr:coverage_percent', 'cr:molecular_weight', 'cr:peptide_count', 'cr:abundance', 'cr:norm_abundance']
    numeric_field = None
    for col in df.columns:
        if col in possible_numeric_fields or ('percent' in col or 'abundance' in col or 'weight' in col):
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
    # If none is numeric, try to convert those columns
    if not numeric_field:
        for possible_col in possible_numeric_fields:
            if possible_col in df.columns:
                try:
                    df[possible_col] = pd.to_numeric(df[possible_col], errors='coerce')
                    if df[possible_col].notnull().sum() > 0:
                        numeric_field = possible_col
                        break
                except Exception:
                    pass
    if not numeric_field:
        print('No obvious numeric field found in columns:', df.columns.tolist())
        # Select the first numeric-looking column if present
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
    print(f"Using numeric field for filtering/normalization: {numeric_field}")

    # Filtering
    threshold = None
    if numeric_field:
        # Set threshold to a low quantile to keep example output
        threshold = df[numeric_field].dropna().quantile(0.70)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a possible categorical field
        group_field = None
        for field in df.columns:
            if field not in [numeric_field, f"{numeric_field}_normalized"] and df[field].dtype == 'O':
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print('No data to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if selected_record_set_id and selected_record_set_id in dataframes and 'filtered_df' in locals():
    # Plot histogram for numeric field
    if numeric_field:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20)
        plt.title(f"Histogram of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    # If grouping, plot comparison
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No filtered data to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the mass spectrometry proteomics dataset using the `mlcroissant` library, listing available record sets and fields through their `@id` identifiers. We extracted data as pandas DataFrames, performed basic filtering and normalization on numeric fields, and visualized their distributions. This workflow demonstrates how to programmatically investigate and process FAIR-compliant datasets described by the Croissant specification, enabling further analysis and reproducible science.

You can now extend this notebook for deeper analyses or to process other record sets or fields as needed.